# 書法單字圖片資料集前處理

這份 notebook 會讀取 `/Users/lainantian/Developer/Caligraphy` 底下的多個 bundle，合併 CSV metadata，篩選指定書體，對應本機圖片路徑，並將圖片前處理成 96 x 96 的灰階 PNG。

預設 `RUN_FULL = False`，只抽樣處理 100 張用於測試。確認結果後，將 `RUN_FULL` 改成 `True` 才會處理完整資料集。

## 1. 匯入套件與設定路徑

設定資料根目錄、輸出目錄、要使用的 bundle、書體映射與抽樣開關。

In [ ]:
from pathlib import Path
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

try:
    from scipy import ndimage as ndi
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

ROOT = Path('/Users/lainantian/Developer/Caligraphy')
OUTPUT_ROOT = Path('/Users/lainantian/Developer/Caligraphy_preprocessed_96')
OUTPUT_IMAGE_ROOT = OUTPUT_ROOT / 'images'

RUN_FULL = False
SAMPLE_SIZE = 100
RANDOM_SEED = 42

IMAGE_SIZE = 96
MIN_AREA = 20

BUNDLE_NAMES = [
    'common_c_bundle',
    'common_g_bundle 2',
    'common_h_bundle',
    'common_i_bundle',
    'common_j_bundle',
    'less_common_b_bundle',
]

ORIGINAL_COLUMNS = [
    'query_char',
    'style_code',
    'style_name',
    'era',
    'author',
    'work_title',
    'detail_title',
    'image_url',
    'thumb_url',
]

STYLE_MAP = {
    '楷书': '楷書',
    '篆书': '篆書',
    '草书': '草書',
    '章草': '草書',
    '行书': '行書',
    '隶书': '隸書',
}
STYLE_LABEL_ORDER = ['楷書', '篆書', '草書', '行書', '隸書']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

print(f'ROOT exists: {ROOT.exists()}')
print(f'OpenCV available: {HAS_CV2}; SciPy available: {HAS_SCIPY}')

## 2. 讀取所有 bundle 的 CSV 並合併 metadata

每個 bundle 只讀取主要 CSV，加入 `bundle` 與 `bundle_path`，再依指定書體做篩選並新增 `style_label`。

In [ ]:
def read_bundle_csv(bundle_name: str) -> pd.DataFrame:
    bundle_path = ROOT / bundle_name
    if not bundle_path.exists():
        raise FileNotFoundError(f'Missing bundle directory: {bundle_path}')

    csv_files = sorted(bundle_path.glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV found in: {bundle_path}')

    csv_path = csv_files[0]
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    df.columns = [str(col).strip().lstrip('\ufeff') for col in df.columns]

    missing_columns = [col for col in ORIGINAL_COLUMNS if col not in df.columns]
    if missing_columns:
        raise ValueError(f'{csv_path} is missing columns: {missing_columns}')

    df = df[ORIGINAL_COLUMNS].copy()
    df['bundle'] = bundle_name
    df['bundle_path'] = str(bundle_path)
    return df


frames = []
for bundle_name in BUNDLE_NAMES:
    frames.append(read_bundle_csv(bundle_name))

metadata_all = pd.concat(frames, ignore_index=True)
metadata = metadata_all[metadata_all['style_name'].isin(STYLE_MAP)].copy()
metadata['style_label'] = metadata['style_name'].map(STYLE_MAP)
metadata['source_index'] = metadata.index

print(f'All metadata rows: {len(metadata_all):,}')
print(f'Filtered metadata rows: {len(metadata):,}')
display(metadata.head())

## 3. 檢查篩選後的書體分布

`章草` 會併入 `草書`；`简牍`、`魏碑` 等其他類別不使用。

In [ ]:
style_counts = metadata['style_label'].value_counts().reindex(STYLE_LABEL_ORDER).fillna(0).astype(int)
display(style_counts.to_frame('count'))

style_source_counts = pd.crosstab(metadata['style_name'], metadata['style_label'])
display(style_source_counts)

## 4. 建立本機圖片索引

掃描每個 bundle 的 `*_downloads/{style_name}/` 目錄，解析檔名中的單字、書體、作者與作品名，建立可與 metadata 對應的圖片索引。

In [ ]:
def norm_text(value) -> str:
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except TypeError:
        pass
    text = unicodedata.normalize('NFKC', str(value)).strip()
    if text.lower() in {'nan', 'none', 'nat'}:
        return ''
    return text


def work_name_for_filename(value) -> str:
    return norm_text(value) or '未标注作品'


def parse_download_filename(path: Path) -> dict:
    parts = path.stem.split('_')
    return {
        'file_serial': parts[0] if len(parts) > 0 else '',
        'file_query_char': parts[1] if len(parts) > 1 else '',
        'file_style_name': parts[2] if len(parts) > 2 else path.parent.name,
        'file_author': parts[3] if len(parts) > 3 else '',
        'file_work_title': '_'.join(parts[4:]) if len(parts) > 4 else '',
    }


download_records = []
for bundle_name in BUNDLE_NAMES:
    bundle_path = ROOT / bundle_name
    download_dirs = sorted(bundle_path.glob('*_downloads'))
    if not download_dirs:
        warnings.warn(f'No downloads directory found in {bundle_path}')
        continue

    download_root = download_dirs[0]
    for style_name in STYLE_MAP:
        style_dir = download_root / style_name
        if not style_dir.exists():
            warnings.warn(f'Missing style directory: {style_dir}')
            continue

        for path in style_dir.rglob('*'):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            parsed = parse_download_filename(path)
            download_records.append({
                'bundle': bundle_name,
                'style_name': style_name,
                'style_label': STYLE_MAP[style_name],
                'image_path': str(path),
                **parsed,
            })

downloads = pd.DataFrame(download_records)

if not downloads.empty:
    downloads['file_serial_num'] = pd.to_numeric(downloads['file_serial'], errors='coerce')
    downloads['_query_char_norm'] = downloads['file_query_char'].map(norm_text)
    downloads['_author_norm'] = downloads['file_author'].map(norm_text)
    downloads['_work_norm'] = downloads['file_work_title'].map(work_name_for_filename)
    downloads = downloads.sort_values(
        ['bundle', 'style_name', '_query_char_norm', '_author_norm', '_work_norm', 'file_serial_num', 'image_path'],
        na_position='last',
    ).reset_index(drop=True)

print(f'Indexed local images: {len(downloads):,}')
display(downloads.head())

## 5. 將 metadata 對應到本機圖片路徑

優先用 `bundle + style_name + query_char + author + work_title + occurrence` 對應；若遇到重複資料或作品欄位不一致，會退回到較寬鬆的 key。找不到圖片的列會保留在檢查表中，但不會進入實際處理。

In [ ]:
def add_metadata_match_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['_query_char_norm'] = out['query_char'].map(norm_text)
    out['_author_norm'] = out['author'].map(norm_text)
    out['_work_norm'] = out['work_title'].map(work_name_for_filename)
    return out


def fill_missing_paths_by_first_match(df: pd.DataFrame, download_df: pd.DataFrame, key_cols: list[str]) -> pd.DataFrame:
    if download_df.empty:
        return df

    missing_mask = df['image_path'].isna()
    if not missing_mask.any():
        return df

    first_matches = download_df.drop_duplicates(key_cols)[key_cols + ['image_path']]
    fallback = df.loc[missing_mask, key_cols].merge(
        first_matches.rename(columns={'image_path': '_fallback_image_path'}),
        on=key_cols,
        how='left',
    )['_fallback_image_path'].to_numpy()

    df.loc[missing_mask, 'image_path'] = fallback
    return df


meta = add_metadata_match_keys(metadata)
exact_cols = ['bundle', 'style_name', '_query_char_norm', '_author_norm', '_work_norm']
char_style_cols = ['bundle', 'style_name', '_query_char_norm']

if downloads.empty:
    metadata_with_paths = meta.copy()
    metadata_with_paths['image_path'] = pd.NA
else:
    meta['_exact_occ'] = meta.groupby(exact_cols, dropna=False).cumcount()
    downloads['_exact_occ'] = downloads.groupby(exact_cols, dropna=False).cumcount()

    exact_downloads = downloads[exact_cols + ['_exact_occ', 'image_path']].copy()
    metadata_with_paths = meta.merge(
        exact_downloads,
        on=exact_cols + ['_exact_occ'],
        how='left',
    )

    metadata_with_paths = fill_missing_paths_by_first_match(metadata_with_paths, downloads, exact_cols)
    metadata_with_paths = fill_missing_paths_by_first_match(metadata_with_paths, downloads, char_style_cols)

metadata_with_paths['image_found'] = metadata_with_paths['image_path'].notna()
found_count = int(metadata_with_paths['image_found'].sum())
missing_count = int((~metadata_with_paths['image_found']).sum())

print(f'Metadata rows with local images: {found_count:,}')
print(f'Metadata rows missing local images: {missing_count:,}')

missing_images = metadata_with_paths[~metadata_with_paths['image_found']].copy()
display(metadata_with_paths.head())
display(missing_images.head())

## 6. 定義影像前處理函式

處理原則：保留灰階筆觸與書寫細節，不做 skeletonize、不輸出 binary image、不做強去噪。Otsu threshold 只用於建立暫時 mask，以偵測小雜點與字本體 bounding box。

In [ ]:
def otsu_threshold(gray: np.ndarray) -> int:
    gray_u8 = np.asarray(gray, dtype=np.uint8)
    hist = np.bincount(gray_u8.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return 127

    bins = np.arange(256, dtype=np.float64)
    weight_bg = np.cumsum(hist)
    weight_fg = total - weight_bg
    sum_bg = np.cumsum(hist * bins)
    sum_total = sum_bg[-1]

    valid = (weight_bg > 0) & (weight_fg > 0)
    between = np.zeros(256, dtype=np.float64)
    mean_bg = np.zeros(256, dtype=np.float64)
    mean_fg = np.zeros(256, dtype=np.float64)
    mean_bg[valid] = sum_bg[valid] / weight_bg[valid]
    mean_fg[valid] = (sum_total - sum_bg[valid]) / weight_fg[valid]
    between[valid] = weight_bg[valid] * weight_fg[valid] * (mean_bg[valid] - mean_fg[valid]) ** 2
    return int(np.argmax(between))


def component_labels(mask: np.ndarray):
    mask_u8 = mask.astype(np.uint8)

    if HAS_CV2:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
        sizes = stats[:, cv2.CC_STAT_AREA]
        return labels, num_labels, sizes

    if HAS_SCIPY:
        labels, num_labels = ndi.label(mask_u8, structure=np.ones((3, 3), dtype=np.uint8))
        sizes = np.bincount(labels.ravel())
        return labels, num_labels + 1, sizes

    warnings.warn('Neither OpenCV nor SciPy is available; small-component cleanup is skipped.')
    return None, 0, None


def remove_small_ink_components(gray: np.ndarray, min_area: int = MIN_AREA) -> np.ndarray:
    threshold = otsu_threshold(gray)
    ink_mask = gray < threshold
    labels, _, sizes = component_labels(ink_mask)
    if labels is None or sizes is None or len(sizes) <= 1:
        return gray.copy()

    label_ids = np.arange(len(sizes))
    small_labels = label_ids[(label_ids != 0) & (sizes < min_area)]
    if len(small_labels) == 0:
        return gray.copy()

    cleaned = gray.copy()
    cleaned[np.isin(labels, small_labels)] = 255
    return cleaned


def find_ink_bbox(gray: np.ndarray, min_area: int = MIN_AREA):
    threshold = otsu_threshold(gray)
    ink_mask = gray < threshold

    labels, _, sizes = component_labels(ink_mask)
    if labels is not None and sizes is not None and len(sizes) > 1:
        label_ids = np.arange(len(sizes))
        keep_labels = label_ids[(label_ids != 0) & (sizes >= min_area)]
        if len(keep_labels) > 0:
            ink_mask = np.isin(labels, keep_labels)

    ys, xs = np.where(ink_mask)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x0 = int(xs.min())
    x1 = int(xs.max()) + 1
    y0 = int(ys.min())
    y1 = int(ys.max()) + 1
    return x0, y0, x1, y1


def crop_pad_resize(gray: np.ndarray, bbox, output_size: int = IMAGE_SIZE, crop_margin_ratio: float = 0.04, square_padding_ratio: float = 0.08) -> Image.Image:
    h, w = gray.shape

    if bbox is None:
        crop = gray
    else:
        x0, y0, x1, y1 = bbox
        box_size = max(x1 - x0, y1 - y0)
        margin = max(2, int(round(box_size * crop_margin_ratio)))
        x0 = max(0, x0 - margin)
        y0 = max(0, y0 - margin)
        x1 = min(w, x1 + margin)
        y1 = min(h, y1 + margin)
        crop = gray[y0:y1, x0:x1]

    if crop.size == 0:
        crop = gray

    ch, cw = crop.shape
    side = max(ch, cw)
    side = int(math.ceil(side * (1 + 2 * square_padding_ratio)))
    side = max(side, ch, cw, 1)

    square = np.full((side, side), 255, dtype=np.uint8)
    y_offset = (side - ch) // 2
    x_offset = (side - cw) // 2
    square[y_offset:y_offset + ch, x_offset:x_offset + cw] = crop

    return Image.fromarray(square, mode='L').resize((output_size, output_size), Image.Resampling.LANCZOS)


def preprocess_image(path, output_size: int = IMAGE_SIZE, min_area: int = MIN_AREA, invert_mean_threshold: float = 127.0) -> Image.Image:
    path = Path(path)
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert('L')
        arr = np.asarray(image, dtype=np.float32)

    # 黑底白字的圖片通常平均亮度偏低；先反相，統一成白底黑字。
    if float(np.mean(arr)) < invert_mean_threshold:
        arr = 255.0 - arr

    # 輕度 contrast normalization：保留灰階筆觸，不做強二值化。
    lo, hi = np.percentile(arr, [1.0, 99.5])
    if hi > lo + 1e-6:
        arr = (arr - lo) * (255.0 / (hi - lo))
    arr = np.clip(arr, 0, 255).astype(np.uint8)

    cleaned = remove_small_ink_components(arr, min_area=min_area)
    bbox = find_ink_bbox(cleaned, min_area=min_area)
    return crop_pad_resize(cleaned, bbox, output_size=output_size)


def safe_filename_part(value, max_len: int = 48) -> str:
    text = norm_text(value)
    if not text:
        text = 'unknown'

    invalid_chars = set('<>:"/|?*') | {chr(92)}
    text = ''.join('_' if ch in invalid_chars or ord(ch) < 32 else ch for ch in text)
    text = re.sub(r'\s+', '_', text)
    text = re.sub(r'_+', '_', text).strip('._ ')
    if not text:
        text = 'unknown'
    return text[:max_len]


print('preprocess_image(path) is ready.')

## 7. 建立輸出資料夾並選擇要處理的資料

`RUN_FULL = False` 時只會從已找到本機圖片的資料中抽樣 100 張。`RUN_FULL = True` 時才會處理全部可用資料。

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
for style_label in STYLE_LABEL_ORDER:
    (OUTPUT_IMAGE_ROOT / style_label).mkdir(parents=True, exist_ok=True)

usable_metadata = metadata_with_paths[metadata_with_paths['image_found']].copy()

if RUN_FULL:
    process_df = usable_metadata.copy()
else:
    process_df = usable_metadata.sample(
        n=min(SAMPLE_SIZE, len(usable_metadata)),
        random_state=RANDOM_SEED,
    ).copy()

process_df = process_df.sort_values('source_index').reset_index(drop=True)

print(f'RUN_FULL = {RUN_FULL}')
print(f'Rows selected for preprocessing: {len(process_df):,}')
display(process_df[['query_char', 'style_name', 'style_label', 'author', 'work_title', 'image_path']].head())

## 8. 執行前處理並輸出 PNG 與 metadata.csv

輸出結構：

```text
Caligraphy_preprocessed_96/
  images/
    楷書/
    篆書/
    草書/
    行書/
    隸書/
  metadata.csv
```

處理失敗的圖片會記錄到 `preprocessing_failures.csv`，不會中斷整批流程。

In [ ]:
def output_filename(row: pd.Series) -> str:
    idx = int(row['source_index'])
    query_char = safe_filename_part(row['query_char'], max_len=12)
    style_label = safe_filename_part(row['style_label'], max_len=12)
    author = safe_filename_part(row['author'], max_len=48)
    return f'{idx:08d}_{query_char}_{style_label}_{author}.png'


processed_records = []
failure_records = []

for _, row in tqdm(process_df.iterrows(), total=len(process_df), desc='Preprocessing images'):
    image_path = row['image_path']
    style_label = row['style_label']
    processed_path = OUTPUT_IMAGE_ROOT / style_label / output_filename(row)

    try:
        image = preprocess_image(image_path)
        image.save(processed_path, format='PNG')

        record = row.to_dict()
        record['processed_path'] = str(processed_path)
        processed_records.append(record)
    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as exc:
        failure_records.append({
            'source_index': row.get('source_index'),
            'bundle': row.get('bundle'),
            'query_char': row.get('query_char'),
            'style_name': row.get('style_name'),
            'style_label': row.get('style_label'),
            'author': row.get('author'),
            'work_title': row.get('work_title'),
            'image_path': image_path,
            'error': repr(exc),
        })

processed_df = pd.DataFrame(processed_records)
failures_df = pd.DataFrame(failure_records)

FINAL_METADATA_COLUMNS = ORIGINAL_COLUMNS + ['bundle', 'image_path', 'processed_path', 'style_label']
if not processed_df.empty:
    processed_metadata = processed_df[FINAL_METADATA_COLUMNS].copy()
else:
    processed_metadata = pd.DataFrame(columns=FINAL_METADATA_COLUMNS)

metadata_csv_path = OUTPUT_ROOT / 'metadata.csv'
processed_metadata.to_csv(metadata_csv_path, index=False, encoding='utf-8-sig')

if not failures_df.empty:
    failures_df.to_csv(OUTPUT_ROOT / 'preprocessing_failures.csv', index=False, encoding='utf-8-sig')

print(f'Processed images: {len(processed_metadata):,}')
print(f'Failures: {len(failures_df):,}')
print(f'Metadata written to: {metadata_csv_path}')
display(processed_metadata.head())
display(failures_df.head())

## 9. 抽樣檢查：顯示處理後資料的書體數量

這裡顯示已處理輸出中的 `style_label` 分布。若 `RUN_FULL = False`，數量就是 100 張抽樣資料的分布。

In [ ]:
processed_style_counts = processed_metadata['style_label'].value_counts().reindex(STYLE_LABEL_ORDER).fillna(0).astype(int)
display(processed_style_counts.to_frame('processed_count'))

## 10. 抽樣檢查：前 30 名作者資料數量

這裡以篩選後且找到本機圖片的 metadata 統計作者數量；可以用來檢查資料是否被少數作者主導。

In [ ]:
author_series = usable_metadata['author'].map(lambda x: norm_text(x) or '未知')
top_authors = author_series.value_counts().head(30)
display(top_authors.to_frame('count'))

## 11. 抽樣檢查：20 張 before / after comparison

隨機顯示 20 張原圖與處理後圖片，檢查是否保留筆觸、飛白與整體字形比例。

In [ ]:
def show_before_after(df: pd.DataFrame, n: int = 20, random_state: int = RANDOM_SEED):
    if df.empty:
        print('No processed images to display.')
        return

    sample = df.sample(n=min(n, len(df)), random_state=random_state).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, max(3, 2.8 * len(sample))))
    if len(sample) == 1:
        axes = np.array([axes])

    for i, row in sample.iterrows():
        before_ax, after_ax = axes[i]
        try:
            before = Image.open(row['image_path']).convert('L')
            after = Image.open(row['processed_path']).convert('L')
        except Exception as exc:
            before_ax.text(0.5, 0.5, f'Load failed: {exc}', ha='center', va='center')
            after_ax.axis('off')
            continue

        before_ax.imshow(before, cmap='gray', vmin=0, vmax=255)
        before_ax.set_title(f"Before | {row['query_char']} | {row['style_name']}")
        before_ax.axis('off')

        after_ax.imshow(after, cmap='gray', vmin=0, vmax=255)
        after_ax.set_title(f"After | {row['style_label']}")
        after_ax.axis('off')

    plt.tight_layout()
    plt.show()


show_before_after(processed_metadata, n=20)